In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

df = pd.read_csv("../../docs/sonar/sonar.all-data.data", header=None)

# TensorBoard para poder ver los gráficos en vivo luego
writer = SummaryWriter("runs")

#Train split
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].map({'M': 1, 'R': 0}).values # M = Mina (1), R = Roca (0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train) 
X_test = scaler.transform(X_test)      

In [6]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

train_dataset = TensorDataset(X_train_t, y_train_t)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

In [7]:
class SonarNet(nn.Module):
    def __init__(self):
        super(SonarNet, self).__init__()
        self.layer1 = nn.Linear(60, 32)
        self.bn1 = nn.BatchNorm1d(32) 
        self.relu1 = nn.ReLU()
        self.dropout = nn.Dropout(0.3) 
        
        self.layer2 = nn.Linear(32, 16)
        self.relu2 = nn.ReLU()
        
        self.output = nn.Linear(16, 1)

    def forward(self, x):
        x = self.relu1(self.bn1(self.layer1(x)))
        x = self.dropout(x)
        x = self.relu2(self.layer2(x))
        x = self.output(x)
        return x

model = SonarNet()

In [8]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

epochs = 150
history_loss = [] 

print("Iniciando adestramento...")
for epoch in range(epochs):
    model.train() 
    epoch_loss = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()            
        y_pred = model(X_batch)     
        loss = criterion(y_pred, y_batch)
        loss.backward()                
        optimizer.step()                 
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    history_loss.append(avg_loss)
    
    writer.add_scalar("Loss/train", avg_loss, epoch)

writer.close()

Iniciando adestramento...


In [9]:
model.eval() 

with torch.no_grad():
    y_test_pred = model(X_test_t) 

probabilidades = torch.sigmoid(y_test_pred)

prediccion_clases = torch.round(probabilidades)

correctos = (prediccion_clases == y_test_t).sum().item()
total = y_test_t.size(0)
accuracy = correctos / total

print("---")
print(f"Resultado final:")
print(f"Precisión (Accuracy) no conxunto de test: {accuracy * 100:.2f}%")

---
Resultado final:
Precisión (Accuracy) no conxunto de test: 85.71%
